In [4]:
import os
import json
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

In [5]:

# ==========================================
# 1. HELPER FUNCTIONS FOR EXTRACTION
# ==========================================
def extract_json_fields(text):
    """
    Safely extracts 'solution' and 'short answer' from LLM outputs.
    Includes fallback regex matching if standard JSON parsing fails.
    """
    if pd.isna(text):
        return None, None
        
    text = str(text).strip()
    
    # Strip markdown code block formatting if present
    if text.startswith('```json'):
        text = text[7:]
    elif text.startswith('```'):
        text = text[3:]
    if text.endswith('```'):
        text = text[:-3]
        
    text = text.strip()

    try:
        data = json.loads(text)
        solution = data.get('solution', '')
        short_ans = data.get('short answer', data.get('short_answer', ''))
        return str(short_ans), str(solution)
    except Exception:
        return "", ""

        # Fallback: regex extraction if JSON is malformed
        short_ans, solution = "", ""
        
        sa_match = re.search(r'"short\s?_?answer"\s*:\s*"?([^"}]+)"?', text, re.IGNORECASE)
        if sa_match: 
            short_ans = sa_match.group(1).strip()
            
        sol_match = re.search(r'"solution"\s*:\s*"?([^"}]+)"?', text, re.IGNORECASE)
        if sol_match: 
            solution = sol_match.group(1).strip()
            
        return short_ans, solution

# ==========================================
# 2. GRADING FUNCTIONS
# ==========================================

def extract_number(text):
    """
    Helper function to extract the first valid number (including decimal 
    and scientific notation like 1.5e-3 or -4.90E+06) from a string.
    """
    # Regex for standard and scientific number formats
    match = re.search(r"[-+]?(?:\d*\.\d+|\d+)(?:[eE][-+]?\d+)?", str(text))
    if match:
        return float(match.group(0))
    return None

def grade_short_answer(prediction, truth, tolerance_percent=0.5):
    """
    Grades short answer by extracting numbers and checking if the prediction 
    is within the specified tolerance_percent of the ground truth.
    """
    try:
        pred_val = extract_number(prediction)
        truth_val = extract_number(truth)
        
        # If either doesn't contain a number, fallback to basic string match
        if pred_val is None or truth_val is None:
            return str(prediction).strip().lower() == str(truth).strip().lower()
            
        # Handle the edge case where the ground truth is exactly 0
        if truth_val == 0:
            return pred_val == 0
            
        # Calculate percentage error and check against tolerance
        error = abs(pred_val - truth_val) / abs(truth_val)
        return error <= (tolerance_percent / 100.0)
        
    except Exception:
        # Fallback to exact match if any unexpected parsing error occurs
        return str(prediction).strip().lower() == str(truth).strip().lower()

def grade_mcq(prediction, truth):
    return str(prediction).strip().lower() == str(truth).strip().lower()

def grade_free_form(prediction, truth):
    return str(prediction).strip().lower() == str(truth).strip().lower()

def evaluate_answer(prediction, truth, question_type):
    if pd.isna(prediction) or pd.isna(truth):
        return 0
        
    q_type = str(question_type).lower()
    
    if "mcq" in q_type:
        is_correct = grade_mcq(prediction, truth)
    elif "short" in q_type:
        is_correct = grade_short_answer(prediction, truth)
    else:
        is_correct = grade_free_form(prediction, truth)
        
    return 1 if is_correct else 0

def highlight_correctness(val):
    """Colors the cell green if 1 (correct), red if 0 (incorrect)."""
    if val == 1:
        return 'background-color: #C6EFCE; color: #006100' # Green
    elif val == 0:
        return 'background-color: #FFC7CE; color: #9C0006' # Red
    return ''

In [6]:
# ==========================================
# 3. MAIN WORKFLOW
# ==========================================
def main():
    print("[INFO] Starting evaluation pipeline...")
    
    out_dir = Path("result/process")
    out_dir.mkdir(parents=True, exist_ok=True)
    
    # Define mapping of response columns to languages
    lang_mapping = {
        'response': 'English',
        'response_arabic': 'Arabic',
        'response_chinese': 'Chinese',
        'response_hindi': 'Hindi',
        'response_indonesian': 'Indonesian',
        'response_kazakh': 'Kazakh',
        'response_swahili': 'Swahili'
    }
    
    # 1. LOAD EVERY FILE
    base_dir = Path("result/raw")
    all_dataframes = []
    
    if not base_dir.exists():
        print(f"[!] Directory {base_dir} not found. Please check your path.")
        return

    print("[INFO] Loading Excel files...")
    for xlsx_file in base_dir.rglob("*.xlsx"):
        parts = xlsx_file.parts
        if len(parts) >= 4:
            subject = parts[-3]
            instance = parts[-2]
            model_name = xlsx_file.stem # Infer model name directly from file name
            
            try:
                df = pd.read_excel(xlsx_file, engine='openpyxl')
                df['subject'] = subject
                df['instance'] = instance
                df['model'] = model_name
                all_dataframes.append(df)
            except Exception as e:
                print(f"[!] Error reading {xlsx_file}: {e}")
                
    if not all_dataframes:
        print("[!] No data loaded. Exiting.")
        return
        
    master_df = pd.concat(all_dataframes, ignore_index=True)
    print(f"[INFO] Loaded {len(master_df)} total rows.")

    # 2. EXTRACT JSON & 3. GRADE EACH PROBLEM
    print("[INFO] Extracting JSON and grading responses...")
    eval_records = []
    
    for index, row in master_df.iterrows():
        truth = row.get('answer', '')
        q_type = row.get('question_type', 'Short Answer')
        model = row['model']
        subject = row['subject'].capitalize()
        problem_id = row.get('problem_id', 'Unknown')
        instance = row.get('instance', 'Unknown')
        
        for col_name, lang_name in lang_mapping.items():
            if col_name in row:
                raw_response = row[col_name]
                short_ans, solution = extract_json_fields(raw_response)
                score = evaluate_answer(short_ans, truth, q_type)
                
                eval_records.append({
                    'Model': model,
                    'Subject': subject,
                    'Language': lang_name,
                    'Problem_ID': problem_id,
                    'Instance': instance,
                    'Accuracy': score
                })

    eval_df = pd.DataFrame(eval_records)

    # 4. CALCULATE METRICS
    print("[INFO] Computing Metrics...")

    # Metric 1: Average Accuracy
    avg_df = eval_df.groupby(['Model', 'Subject', 'Language'])['Accuracy'].mean().reset_index()
    avg_pivot = avg_df.pivot_table(index='Model', columns=['Subject', 'Language'], values='Accuracy')

    # Metric 2: Worst-Case Accuracy
    # Group by Problem_ID first to find if *any* instance failed (min = 0 if any failed)
    wc_prob_df = eval_df.groupby(['Model', 'Subject', 'Language', 'Problem_ID'])['Accuracy'].min().reset_index()
    wc_df = wc_prob_df.groupby(['Model', 'Subject', 'Language'])['Accuracy'].mean().reset_index()
    wc_pivot = wc_df.pivot_table(index='Model', columns=['Subject', 'Language'], values='Accuracy')

    # Metric 3: Reasoning Robustness
    # safely handle 0/0 (NaN) and 1/0 (inf)
    robustness_pivot = (wc_pivot / avg_pivot).replace([np.inf, -np.inf], np.nan).fillna(0)

    # 5. EXPORT ARTIFACTS
    print(f"[INFO] Saving artifacts to {out_dir}...")
    
    avg_acc_path = out_dir / "average_accuracy.xlsx"
    wc_acc_path = out_dir / "worst_case_accuracy.xlsx"
    
    # Apply .round(2) right as you export to Excel
    avg_pivot.round(2).to_excel(avg_acc_path)
    wc_pivot.round(2).to_excel(wc_acc_path)
    
    print(f"  -> Saved {avg_acc_path}")
    print(f"  -> Saved {wc_acc_path}")

    # ==========================================
    # 6. EXPORT DETAILED 'ANALYZED' FOLDERS
    # ==========================================
    print("[INFO] Generating detailed analyzed spreadsheets...")
    analyzed_dir = out_dir / "analyzed"
    
    # Pivot the data: Rows = Problem/Language, Columns = Instances
    detail_pivot = eval_df.pivot_table(
        index=['Model', 'Subject', 'Language', 'Problem_ID'],
        columns='Instance',
        values='Accuracy'
    ).reset_index()

    # Identify which columns are the "instance" columns dynamically
    instance_cols = [col for col in detail_pivot.columns if col not in ['Model', 'Subject', 'Language', 'Problem_ID']]

    # Loop through Models and Subjects to create files
    for model_name, model_df in detail_pivot.groupby('Model'):
        model_dir = analyzed_dir / model_name
        model_dir.mkdir(parents=True, exist_ok=True)
        
        for subject, subj_df in model_df.groupby('Subject'):
            file_path = model_dir / f"{subject.lower()}.xlsx"
            
            # Use ExcelWriter to allow multiple sheets in the same file
            with pd.ExcelWriter(file_path, engine='openpyxl') as writer:
                
                # Group by Language to create a new sheet for each
                for language, lang_df in subj_df.groupby('Language'):
                    
                    # Drop columns that are no longer needed for display
                    out_df = lang_df.drop(columns=['Model', 'Subject', 'Language'])
                    
                    # Apply the red/green colors just to the instance columns
                    # Note: If you use Pandas older than 2.1.0, change .map to .applymap
                    styled_df = out_df.style.map(highlight_correctness, subset=instance_cols)
                    
                    # Save the dataframe to its specific language sheet
                    styled_df.to_excel(writer, sheet_name=language, index=False)
            
    print(f"  -> Saved detailed spreadsheets to {analyzed_dir}")

if __name__ == "__main__":
    main()

[INFO] Starting evaluation pipeline...
[INFO] Loading Excel files...
[INFO] Loaded 18810 total rows.
[INFO] Extracting JSON and grading responses...
[INFO] Computing Metrics...
[INFO] Saving artifacts to result\process...
  -> Saved result\process\average_accuracy.xlsx
  -> Saved result\process\worst_case_accuracy.xlsx
[INFO] Generating detailed analyzed spreadsheets...
  -> Saved detailed spreadsheets to result\process\analyzed
